# Real-time Buffer Processing - Sliding Window

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

This notebook demonstrates a **sliding buffer** for real-time EEG processing simulation. We iterate through the P4 channel signal in chunks of 50 samples, maintaining a sliding buffer of 500 samples (2.5 seconds). At each step, we apply a simple moving average filter (window=11) to the buffer and record the filtered output.

## What this notebook does

1. Loads the P4 channel from the local EEG dataset
2. Simulates real-time processing in chunks of 50 samples
3. Maintains a sliding buffer of 500 samples (2.5 seconds)
4. Applies a moving average filter (window=11) at each step
5. Records the progressive filtered output

## What you should expect to see

- The top plot shows the **original raw signal** (first 5000 samples) in blue
- The bottom plot shows the **real-time filtered output** in red, demonstrating progressive filtering
- A vertical dashed line marks the "current" position at sample 2500
- The filtered output is smoother than the original, showing the effect of the moving average

## Key parameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| FS | 200 Hz | Sampling rate |
| CHUNK_SIZE | 50 | Samples per processing chunk |
| BUFFER_SIZE | 500 | Sliding buffer length (2.5 s) |
| MA_WINDOW | 11 | Moving average window size |
| N_PLOT | 5000 | Number of samples to plot |

## 1. Install dependencies

In [ ]:
!pip install scipy numpy plotly wfdb

## 2. Clone repo and download data

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1

## 3. Load the EEG signal

We load subject 1, experiment 1, session 2, channel **P4** (parietal region).

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')

## 4. Apply the analysis

We simulate real-time processing by iterating through the signal in chunks of 50 samples. A sliding buffer of 500 samples (2.5 seconds) is maintained, and a moving average filter (window=11) is applied at each step.

In [ ]:
CHUNK_SIZE = 50
BUFFER_SIZE = 500
MA_WINDOW = 11
N_PLOT = 5000
CURRENT_POS = 2500

n_samples = min(N_PLOT, len(channel_data))
signal_plot = channel_data[:n_samples]

def moving_average(data, window):
    if len(data) < window:
        return data.copy()
    kernel = np.ones(window) / window
    padded = np.pad(data, (window // 2, window - 1 - window // 2), mode='edge')
    return np.convolve(padded, kernel, mode='valid')

buffer = np.zeros(BUFFER_SIZE)
buffer_fill = 0
filtered_output = np.zeros(n_samples)

for start in range(0, n_samples, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, n_samples)
    chunk = signal_plot[start:end]
    for i, sample in enumerate(chunk):
        if buffer_fill < BUFFER_SIZE:
            buffer[buffer_fill] = sample
            buffer_fill += 1
        else:
            buffer = np.roll(buffer, -1)
            buffer[-1] = sample
        filtered_buffer = moving_average(buffer[:buffer_fill], MA_WINDOW)
        filtered_output[start + i] = filtered_buffer[-1]

print(f'Processed {n_samples} samples in chunks of {CHUNK_SIZE}')
print(f'Buffer size: {BUFFER_SIZE} samples ({BUFFER_SIZE/fs:.1f} seconds)')
print(f'Moving average window: {MA_WINDOW}')

## 5. Interactive plot

**What to look for:**
- The top plot shows the original raw signal with all its noise and fluctuations
- The bottom plot shows the progressively filtered output — notice how it is smoother
- The vertical dashed line at sample 2500 represents the "current" processing position
- The moving average filter reduces high-frequency noise while preserving the overall trend

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

x = np.arange(n_samples)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original Signal (P4)', 'Real-time Filtered Output (Moving Average)'))

fig.add_trace(go.Scatter(x=x, y=signal_plot, name='Original',
                         line=dict(color='blue', width=0.5)), row=1, col=1)
fig.add_vline(x=CURRENT_POS, line_dash='dash', line_color='black',
              annotation_text='Current position', row=1, col=1)

fig.add_trace(go.Scatter(x=x, y=filtered_output, name='Filtered',
                         line=dict(color='red', width=0.5)), row=2, col=1)
fig.add_vline(x=CURRENT_POS, line_dash='dash', line_color='black',
              annotation_text='Current position', row=2, col=1)

fig.update_layout(height=600, title_text='Real-time Buffer Processing - Sliding Window',
                  xaxis2_title='Sample index', yaxis_title='Amplitude (uV)',
                  yaxis2_title='Amplitude (uV)')
fig.show()

## What did we learn?

- A **sliding buffer** allows real-time processing by keeping a fixed-size window of recent samples
- Processing in **chunks** (50 samples) simulates how real-time systems handle incoming data
- The **moving average filter** smooths the signal by averaging neighboring samples
- The buffer fills up gradually, so early samples have less averaging effect
- This approach is the foundation for real-time EEG monitoring systems